# LipPY Audio CTC v2 — PLAN A.2: Whisper-medium + ANTI-COLLAPSE SpecAug + 10 epoch

**Medium'a geri donus** (large-v3 PLAN B basarisiz, %56 plateau).

PLAN A medium (5 ep, agresif SpecAug F=27 n=2) → PER %32.1 (regression vs v1 %31.2).
Sebep: F=27 + n=2 = potansiyel %67 mel mask → underfit.

**PLAN A.2 fix paket** (ilimli SpecAug + uzun egitim):

| | PLAN A (32.1%) | **PLAN A.2 (hedef %27-30)** |
|---|---|---|
| Model | medium | medium (ayni) |
| EPOCHS | 5 | **10** (daha cok converge) |
| LR_FE | 5e-6 | 5e-6 (ayni) |
| LR_BE | 5e-5 | **1e-4** (CTC head hizli) |
| BS / ACCUM | 4 / 4 | 4 / 4 (ayni) |
| SPEC_FREQ_MASK | **27 (agresif)** | **15 (ilimli)** |
| SPEC_NUM_MASKS | **2** | **1** |
| FORCE_SCRATCH | False | **True** (sifirdan) |
| Mel cache | medium-80 | medium-80 (ayni) |

## Mimari

```
Audio (16kHz, dynamic length 1-10s)
  -> [TRAIN] Mel cache load (~3-5ms) veya WhisperFeatureExtractor fallback
  -> [TRAIN] Mel-domain time stretch (speed perturb) + ILIMLI SpecAugment
  -> WhisperEncoder.from_pretrained('openai/whisper-medium')
     - 24 transformer layer, hidden=1024
     - cikti: (B, T_mel/2, 1024)
     - Ust 6 layer trainable (18 frozen, ~50M param)
  -> Dropout(0.1) + Linear (1024 -> 31 fonem)
  -> CTC loss (input_length = real_frames)
```

## Inputs (Kaggle)

- `brokyewing/lippy-audio-segments` (wav fallback)
- `brokyewing/lippy-test` (text + splits)
- `brokyewing/lippy-mel-cache-whisper-medium` ← medium 80-bin cache

## Cikti

- `KULLANICI/lippy-audio-whisper-ctc-v2-checkpoint` (yeni version)
- Beklenen PER: **%27-30** (PLAN A 32.1%'den -2 ila -5 puan)

## Karar agaci (egitim sonrasi)

- PER ≤ %30 → v3 AV-Fusion'da bu **yeni ckpt'yi** kullan
- PER > %30 → **eski PLAN A %32.1 ckpt'sine geri don** (lokalde var), v3'u o ile yap

## Sonraki adim

PLAN A.2 medium bittikten sonra → **v3 AV-Fusion notebook'u** (LoRA + symmetric fusion + hybrid CTC+attention head).

In [ ]:
!pip install -q transformers torchaudio soundfile

import os, glob, json, shutil, time, math, random, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import WhisperModel, WhisperFeatureExtractor

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'CPU')

# =================== CONFIG (v2 — PLAN A.2: medium + ANTI-COLLAPSE SpecAug + 10 epoch) ===================
# PLAN A.2 mantik:
#   - PLAN A medium (5 ep, agresif SpecAug F=27 n=2) -> PER 32.1% (regression vs v1 31.2%)
#   - Large-v3 plateau (%56) basarisiz, medium'a geri donus
#   - Sebep: F=27 + n=2 = potansiyel %67 mel mask -> underfit
#   - PLAN A.2 = medium + ilimli SpecAug (F=15 n=1) + 10 epoch + LR_BE 1e-4
#   - Beklenen: PER %27-30 (PLAN A 32%'ten -2 ila -5 puan)
HF_MODEL = "openai/whisper-medium"       # PLAN A.2: 769M, hidden=1024, 24 layer (geri donus)

EPOCHS = 10                               # PLAN A.2: 5 -> 10 (medium daha cok converge ister)
LR_FE = 5e-6                              # PLAN A.2: same (medium icin v1 ile ayni)
LR_BE = 1e-4                              # PLAN A.2: 5e-5 -> 1e-4 (CTC head hizli ogrenir)
WARMUP = 1                                # PLAN A.2: ep1 agresif basla
GCLIP = 2.0
ACCUM = 4                                 # PLAN A.2: medium BS=4 fits, effective 16
BS = 4                                    # PLAN A.2: medium ~3GB, T4'te rahat
UPLOAD_EVERY = 2

# Whisper sabit referanslar
WHISPER_MAX_INPUT_SEC = 30
WHISPER_FRAMES_PER_SEC = 50
WHISPER_OUTPUT_FRAMES_MAX = 1500
WHISPER_MEL_HOP = 160
WHISPER_MAX_MEL_FRAMES = 3000

# Dynamic batching (v1 ile ayni)
USE_DYNAMIC_BATCHING = True
DYN_MEL_MULTIPLE = 2

# v2 YENI: SpecAugment (Park 2019, ses icin RNN-T standardi)
# PLAN A.2 ANTI-COLLAPSE: F=27 (80 mel'in %34'u) + n=2 = potansiyel %67 mask
# ilk PLAN A run'da agresif idi, underfit yaratti. F=15 n=1 ile %18 max mask -> ilimli.
USE_SPECAUGMENT = True
SPEC_FREQ_MASK_PARAM = 15                 # PLAN A.2: 27 -> 15 (80 mel'in %18'i)
SPEC_TIME_MASK_PARAM = 70                 # max 70 mel frame mask
SPEC_NUM_MASKS = 1                        # PLAN A.2: 2 -> 1 (toplam mask az)

# v2 YENI: Speed perturbation (Ko 2015, ASR augmentation)
USE_SPEED_PERTURB = True                  # Mel cache ile maliyetsiz
SPEED_VALUES = [0.9, 1.0, 1.1]

# Audio constraints
SAMPLE_RATE = 16000
MIN_AUDIO_SEC = 0.5
MAX_AUDIO_SEC = 10.0

# Freeze
FREEZE_FE = True                          # Conv layer donmus
FREEZE_FIRST_N = 18                       # PLAN A.2: medium 24 layer, ust 6 trainable (~50M param)

# PLAN A.2: Sifirdan basla (large-v3 ckpt uyumsuz, ust uste yazilamaz)
FORCE_SCRATCH = True                      # PLAN A.2: medium + sifirdan
MAX_RESUME_PER = 0.95

# Medium mel cache (80-bin) -- precompute_mel.ipynb medium versiyonunun ciktisi
MEL_CACHE_SLUG = 'lippy-mel-cache-whisper-medium'

print(f'Model: {HF_MODEL}')
print(f'Egitim: EPOCHS={EPOCHS}, BS={BS}xACCUM={ACCUM} = efektif {BS*ACCUM}')
print(f'LR: FE={LR_FE}, BE={LR_BE}  [PLAN A.2: LR_BE 1e-4 anti-collapse]')
print(f'WARMUP: {WARMUP}')
print(f'Freeze: feature_extractor={FREEZE_FE}, ilk {FREEZE_FIRST_N}/24 encoder layer')
print(f'SpecAugment: {USE_SPECAUGMENT} (freq<={SPEC_FREQ_MASK_PARAM}, time<={SPEC_TIME_MASK_PARAM}, n={SPEC_NUM_MASKS})  [PLAN A.2 ilimli]')
print(f'Speed perturb: {USE_SPEED_PERTURB} {SPEED_VALUES if USE_SPEED_PERTURB else ""}')
print(f'Audio: {SAMPLE_RATE}Hz, {MIN_AUDIO_SEC}-{MAX_AUDIO_SEC}s, Dynamic batching: {USE_DYNAMIC_BATCHING}')
print(f'Mel cache slug: {MEL_CACHE_SLUG}')
print(f'FORCE_SCRATCH: {FORCE_SCRATCH}  [PLAN A.2: sifirdan]')


## 1) Vocab — 31 Fonem (v1c + audio-wav2vec ile AYNI)

In [ ]:
PHONEMES = [
    'SIL', 'SPACE',
    'a', 'e', 'i', 'o', 'u', '\u0131', '\u00f6', '\u00fc',
    'b', 'p', 'm', 'f', 'v', 'd', 't', 'n', 'l',
    's', 'z', 'c', '\u00e7', '\u015f', 'j',
    'k', 'g', '\u011f', 'r', 'h', 'y',
]
BLANK_IDX = 0
SPACE_IDX = 1
PH_TO_IDX = {p: i for i, p in enumerate(PHONEMES)}
IDX_TO_PH = {i: p for i, p in enumerate(PHONEMES)}
NUM_PHONEMES = len(PHONEMES)
assert NUM_PHONEMES == 31

CHAR_TO_PH = {ph: ph for ph in PHONEMES if ph not in ('SIL', 'SPACE')}


def text_to_phonemes(text):
    phs = []
    for ch in text.lower().strip():
        if ch == ' ':
            if not phs or phs[-1] != SPACE_IDX:
                phs.append(SPACE_IDX)
        elif ch in CHAR_TO_PH:
            phs.append(PH_TO_IDX[CHAR_TO_PH[ch]])
    while phs and phs[0] == SPACE_IDX: phs.pop(0)
    while phs and phs[-1] == SPACE_IDX: phs.pop()
    return phs


def phonemes_to_text(indices):
    if isinstance(indices, torch.Tensor):
        indices = indices.tolist()
    out = []
    for i in indices:
        if i == BLANK_IDX: continue
        out.append(' ' if i == SPACE_IDX else IDX_TO_PH.get(i, ''))
    return ''.join(out)


print(f'Vocab: {NUM_PHONEMES} fonem')


## 2) Model -- Whisper Encoder + CTC Head (dynamic length)

- WhisperModel.encoder kullanilir, decoder atilir
- Feature extractor (conv1 + conv2 + positional_embedding) donmus
- Ilk 18 transformer layer donmus (Whisper-medium toplam 24 layer)
- Ust 6 transformer layer fine-tune (~50M trainable param)
- CTC head: Linear 1024 -> 31
- Encoder dropout sabit 0.1 (HF Whisper config 0.0 idi, no-op olurdu — fix uygulandi)

**Dynamic length:** HF default encoder TAM 3000 mel frame bekler. Burada custom forward
ile positional embedding `T_out` kadar dilimlenir -> degisken uzunluk destekli.
Padded kisim sessizlik (zero), CTC `input_lengths` gercek frame'leri kullanir.

**Medium mel:** 80 bin input. FP32 yukleme zorunlu degil ama tutarlilik icin
`torch_dtype=torch.float32` kullaniliyor.

In [ ]:
class WhisperAudioModel(nn.Module):
    def __init__(self, model_name=HF_MODEL, num_phonemes=NUM_PHONEMES,
                 freeze_fe=FREEZE_FE, freeze_first_n=FREEZE_FIRST_N,
                 dropout=0.1):
        super().__init__()
        # FIX (v2): Whisper-large-v3 HF safetensors FP16 yuklenebilir
        # -> scaler.unscale_() FP16 gradients reddeder. Force FP32 yukle.
        full_model = WhisperModel.from_pretrained(model_name, torch_dtype=torch.float32)
        self.encoder = full_model.encoder
        del full_model  # decoder atildi, RAM tasarruflu
        torch.cuda.empty_cache()

        hidden = self.encoder.config.d_model
        n_layers = len(self.encoder.layers)
        # FIX (v2): self.encoder.dropout Whisper-large-v3'te float 0.0 -> F.dropout no-op
        # idi. Buyuk model + kucuk veri icin kasitli regularization gerekli.
        self.encoder_dropout_p = dropout
        print(f'Whisper encoder: hidden={hidden}, layers={n_layers}')

        # Freeze feature extractor (conv1 + conv2 + positional)
        if freeze_fe:
            for p in self.encoder.conv1.parameters():
                p.requires_grad = False
            for p in self.encoder.conv2.parameters():
                p.requires_grad = False
            self.encoder.embed_positions.requires_grad = False
            print(f'Conv1 + Conv2 + positional embedding DONMUS')

        # Ilk N transformer layer donmus
        for i, layer in enumerate(self.encoder.layers):
            if i < freeze_first_n:
                for p in layer.parameters():
                    p.requires_grad = False
        print(f'Ilk {freeze_first_n}/{n_layers} transformer layer DONMUS')

        # CTC head (yeni, scratch)
        self.dropout = nn.Dropout(dropout)
        self.ctc_fc = nn.Linear(hidden, num_phonemes)

        # Trainable parametre
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'Parametre: trainable={trainable:,} / total={total:,} '
              f'({100*trainable/total:.1f}%)')

    def _encoder_forward_dynamic(self, input_features):
        """
        Variable-length Whisper encoder forward.
        input_features: (B, 80, T_mel) -- T_mel EVEN, <= 3000.

        HF WhisperEncoder normalde TAM 3000 frame bekler (positional embedding 1500).
        Bu metot pos embed'i T_out kadar dilimleyerek variable length destekler.
        """
        # Conv1 (stride 1) + Conv2 (stride 2)
        x = F.gelu(self.encoder.conv1(input_features))
        x = F.gelu(self.encoder.conv2(x))
        x = x.permute(0, 2, 1)  # (B, T_out, hidden), T_out = T_mel/2

        T_out = x.shape[1]
        # Pos embed dilimle (max 1500 pozisyon var)
        embed_pos = self.encoder.embed_positions.weight[:T_out]
        x = x + embed_pos

        # FIX (v2): encoder.dropout 0.0 idi, sabit p=encoder_dropout_p kullan
        x = F.dropout(x, p=self.encoder_dropout_p, training=self.training)

        # Transformer layers
        for layer in self.encoder.layers:
            x = layer(x, None, None)[0]

        x = self.encoder.layer_norm(x)
        return x  # (B, T_out, hidden)

    def forward(self, input_features):
        """
        input_features: (B, 80, T_mel) -- T_mel <= 3000, EVEN.
        Return: log_probs (T_out, B, V) for CTC, T_out = T_mel/2
        """
        h = self._encoder_forward_dynamic(input_features)
        x = self.dropout(h)
        logits = self.ctc_fc(x)
        log_probs = logits.log_softmax(-1)
        return log_probs.permute(1, 0, 2)  # (T, B, V)

    @staticmethod
    def get_output_length(audio_samples):
        """
        Real audio sample sayisindan Whisper encoder cikti uzunlugunu hesapla.
        Conv2 stride 2 -> output = mel_frames / 2.
        mel_frames = ceil(audio_samples / 160).
        """
        mel_frames = (audio_samples + WHISPER_MEL_HOP - 1) // WHISPER_MEL_HOP
        out_frames = mel_frames // 2
        return min(out_frames, WHISPER_OUTPUT_FRAMES_MAX)


## 3) Veri — Audio + Text, Speaker-Disjoint Split

WhisperFeatureExtractor 16kHz audio'yu log-mel spectrogram'a cevirir (80 bin x 3000 frame).
30 sec'den kisa audio otomatik padded.


In [ ]:
# Auto-discovery (audio v1 ile ayni)
AUDIO_CANDS = (
    glob.glob('/kaggle/input/**/lippy-audio-segments/**/audios', recursive=True)
    + glob.glob('/kaggle/input/**/audios', recursive=True)
)
AUDIO_DIR = AUDIO_CANDS[0] if AUDIO_CANDS else '/kaggle/input/lippy-audio-segments/audios'

TEXT_CANDS = glob.glob('/kaggle/input/**/dataset/texts', recursive=True)
TEXT_DIR = TEXT_CANDS[0] if TEXT_CANDS else '/kaggle/input/lippy-test/dataset/texts'

SPLIT_CANDS = glob.glob('/kaggle/input/**/dataset/splits', recursive=True)
SPLIT_DIR = SPLIT_CANDS[0] if SPLIT_CANDS else '/kaggle/input/lippy-test/dataset/splits'

print(f'AUDIO_DIR: {AUDIO_DIR}')
print(f'TEXT_DIR:  {TEXT_DIR}')
print(f'SPLIT_DIR: {SPLIT_DIR}')

# PLAN B: Mel cache auto-discovery -- FAST PATH (recursive glob YOK)
# Diagnostic'te dogrulandi: /kaggle/input/datasets/brokyewing/lippy-mel-cache-whisper-medium/*.pt
# Bilinen path'leri os.path.isdir ile O(1) kontrol et, yavas glob'a dusme.

def _find_mel_cache():
    """Bilinen Kaggle path pattern'lerini sirayla dene. Bulundugunda directory dondur."""
    user = os.environ.get('KAGGLE_USERNAME', 'brokyewing')
    candidates = [
        # Kaggle dataset mount yapilari (en yaygindan en az yaygin sirali)
        f'/kaggle/input/datasets/{user}/{MEL_CACHE_SLUG}',
        f'/kaggle/input/{MEL_CACHE_SLUG}',
        f'/kaggle/input/datasets/{user}/{MEL_CACHE_SLUG}/mel_cache',
        f'/kaggle/input/{MEL_CACHE_SLUG}/mel_cache',
    ]
    for cand in candidates:
        if os.path.isdir(cand):
            # Sample 1 .pt var mi? (scandir, glob'dan cok hizli)
            try:
                for entry in os.scandir(cand):
                    if entry.name.endswith('.pt'):
                        return cand
                    if entry.is_dir() and entry.name == 'mel_cache':
                        # mel_cache subfolder var, ona gec
                        for sub in os.scandir(entry.path):
                            if sub.name.endswith('.pt'):
                                return entry.path
            except (PermissionError, OSError):
                continue
    return None

MEL_CACHE_DIR = _find_mel_cache()
USE_MEL_CACHE = MEL_CACHE_DIR is not None
if USE_MEL_CACHE:
    # Hizli sayma: os.scandir, glob.glob'tan 5-10x hizli
    cache_count = sum(1 for f in os.scandir(MEL_CACHE_DIR) if f.name.endswith('.pt'))
    print(f'Mel cache: AKTIF -> {MEL_CACHE_DIR} ({cache_count:,} dosya)')
    print(f'  (dataloader 15-25x hizli, on-the-fly Whisper FE atlanir)')
else:
    print(f'Mel cache: YOK ({MEL_CACHE_SLUG} input olarak ekli degil ya da .pt bulunamadi)')
    print(f'  Aranan path\'ler: /kaggle/input/datasets/<user>/{MEL_CACHE_SLUG} ve varyantlari')
    print(f'  Fallback: on-the-fly Whisper FE (yavas, 5-6h/epoch)')

# Whisper feature extractor (fallback path icin gerekli, cache miss durumlarinda)
feature_extractor = WhisperFeatureExtractor.from_pretrained(HF_MODEL)
N_MELS = feature_extractor.feature_size
print(f'Whisper feature extractor: sr={feature_extractor.sampling_rate}, n_mels={N_MELS}')


In [ ]:
def apply_specaugment(mel_features, freq_param=SPEC_FREQ_MASK_PARAM,
                       time_param=SPEC_TIME_MASK_PARAM, n_masks=SPEC_NUM_MASKS):
    """SpecAugment: frequency masking + time masking on mel-spectrogram.
    Park et al. (2019) Interspeech.
    mel_features: (80, T_mel) torch.Tensor
    """
    mel = mel_features.clone()
    F_dim, T_dim = mel.shape

    for _ in range(n_masks):
        # Frequency mask
        f = random.randint(0, freq_param)
        if f > 0 and f < F_dim:
            f_start = random.randint(0, F_dim - f)
            mel[f_start:f_start + f, :] = 0.0

        # Time mask
        t = random.randint(0, min(time_param, T_dim - 1))
        if t > 0:
            t_start = random.randint(0, T_dim - t)
            mel[:, t_start:t_start + t] = 0.0

    return mel


def apply_speed_perturb(wav_tensor, sr=SAMPLE_RATE, speeds=SPEED_VALUES):
    """Speed perturbation via single resample. Ko et al. (2015) Interspeech.
    Audio-domain (pitch + length degisir). Sadece cache miss fallback'inde kullanilir.
    """
    speed = random.choice(speeds)
    if speed == 1.0:
        return wav_tensor
    new_sr = int(sr / speed)
    w = wav_tensor.unsqueeze(0)
    w = torchaudio.functional.resample(w, sr, new_sr)
    return w.squeeze(0)


def apply_mel_time_stretch(mel, speeds=SPEED_VALUES):
    """PLAN B: Mel-domain speed perturbation via F.interpolate (linear, time axis).
    Audio-domain'den farkli: pitch SABIT, sadece length degisir. Approximate (%80-90
    etkili literature gore) ama cache ile maliyetsiz (audio resample yok).

    mel: (80, T_mel) torch.Tensor
    Returns: (80, new_T) torch.Tensor
    """
    speed = random.choice(speeds)
    if speed == 1.0:
        return mel
    T_mel = mel.shape[1]
    new_T = max(2, int(T_mel / speed))   # speed=0.9 -> uzar, speed=1.1 -> kisalir
    # (80, T_mel) -> (1, 80, T_mel) for F.interpolate
    stretched = F.interpolate(
        mel.unsqueeze(0), size=new_T,
        mode='linear', align_corners=False,
    ).squeeze(0)
    return stretched


class WhisperAudioDataset(Dataset):
    def __init__(self, split, fe):
        self.train = split == 'train'
        self.fe = fe
        split_path = os.path.join(SPLIT_DIR, f'{split}.txt')
        with open(split_path, 'r', encoding='utf-8') as f:
            all_ids = [l.strip() for l in f if l.strip()]

        self.ids = []
        skipped_no_audio = 0
        skipped_no_text = 0
        for s in all_ids:
            # PLAN B: cache varsa cache file kontrol et (hizli), yoksa wav kontrol
            if USE_MEL_CACHE:
                exists_input = os.path.exists(f'{MEL_CACHE_DIR}/{s}.pt') or os.path.exists(f'{AUDIO_DIR}/{s}.wav')
            else:
                exists_input = os.path.exists(f'{AUDIO_DIR}/{s}.wav')
            txt_p = os.path.join(TEXT_DIR, f'{s}.txt')
            if not exists_input:
                skipped_no_audio += 1; continue
            if not os.path.exists(txt_p):
                skipped_no_text += 1; continue
            self.ids.append(s)

        kept = 100.0 * len(self.ids) / max(len(all_ids), 1)
        print(f'  [{split}] {len(self.ids)}/{len(all_ids)} ({kept:.1f}%) | '
              f'audio_yok={skipped_no_audio}, text_yok={skipped_no_text}')

    def __len__(self):
        return len(self.ids)

    def _load_mel_from_cache(self, s):
        """PLAN B fast path: cache'ten mel oku (~3-5ms). None donerse cache miss."""
        mel_path = f'{MEL_CACHE_DIR}/{s}.pt'
        if not os.path.exists(mel_path):
            return None
        try:
            mel = torch.load(mel_path, weights_only=True).float()  # FP16 -> FP32
            return mel  # (80, T_mel)
        except Exception:
            return None

    def _load_mel_from_wav(self, s):
        """Fallback: wav -> resample -> speed_perturb -> Whisper FE (~80-110ms).
        Sadece cache miss durumunda kullanilir.
        """
        wav_p = os.path.join(AUDIO_DIR, f'{s}.wav')
        wav, sr = torchaudio.load(wav_p)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] > 1:
            wav = wav.mean(0, keepdim=True)
        wav = wav.squeeze(0).float()

        # Audio-domain speed perturb (fallback path)
        if self.train and USE_SPEED_PERTURB:
            wav = apply_speed_perturb(wav, SAMPLE_RATE, SPEED_VALUES)

        wav_np = wav.numpy()
        n_samples = len(wav_np)
        if n_samples < SAMPLE_RATE * MIN_AUDIO_SEC:
            return None
        if n_samples > SAMPLE_RATE * MAX_AUDIO_SEC:
            if self.train and USE_SPEED_PERTURB:
                return None  # uzayan ses skip
            target = int(SAMPLE_RATE * MAX_AUDIO_SEC)
            if self.train:
                start = random.randint(0, n_samples - target)
            else:
                start = (n_samples - target) // 2
            wav_np = wav_np[start:start + target]

        features = self.fe(
            wav_np, sampling_rate=SAMPLE_RATE, return_tensors='pt',
            padding=False, return_attention_mask=False,
        ).input_features
        return features.squeeze(0).float()  # (80, T_mel)

    def __getitem__(self, i):
        for offset in range(len(self.ids)):
            s = self.ids[(i + offset) % len(self.ids)]
            try:
                # PLAN B: Cache fast path
                features = None
                used_cache = False
                if USE_MEL_CACHE:
                    features = self._load_mel_from_cache(s)
                    used_cache = features is not None

                # Cache miss -> wav fallback
                if features is None:
                    features = self._load_mel_from_wav(s)
                    if features is None:
                        continue

                # PLAN B: Mel-domain speed perturb (sadece cache yolu, hizli)
                # Cache miss durumunda audio-domain zaten yapildi yukarida.
                if used_cache and self.train and USE_SPEED_PERTURB:
                    features = apply_mel_time_stretch(features, SPEED_VALUES)

                T_mel_real = features.shape[1]

                # Cap
                if T_mel_real > WHISPER_MAX_MEL_FRAMES:
                    features = features[:, :WHISPER_MAX_MEL_FRAMES]
                    T_mel_real = WHISPER_MAX_MEL_FRAMES

                # Even-length zorlamasi (Conv2 stride 2 icin)
                if T_mel_real % 2 != 0:
                    features = features[:, :-1]
                    T_mel_real -= 1

                # SpecAugment on mel (train only)
                if self.train and USE_SPECAUGMENT:
                    features = apply_specaugment(features)

                real_frames = T_mel_real // 2

                # Text
                txt_p = os.path.join(TEXT_DIR, f'{s}.txt')
                with open(txt_p, 'r', encoding='utf-8') as f:
                    txt = f.read().strip()
                lab = torch.tensor(text_to_phonemes(txt), dtype=torch.long)
                if len(lab) == 0:
                    continue

                # CTC safety: real_frames >= 2 * text_length
                if real_frames < 2 * len(lab):
                    continue

                return features, lab, real_frames, T_mel_real
            except Exception:
                continue
        raise RuntimeError('Audio yuklenemedi')


def collate(batch):
    """Dynamic batching collate (v1 ile ayni)."""
    feats_list, labs, real_lens, T_mel_list = zip(*batch)
    B = len(feats_list)

    T_mel_batch = max(T_mel_list)
    if T_mel_batch % DYN_MEL_MULTIPLE != 0:
        T_mel_batch += DYN_MEL_MULTIPLE - (T_mel_batch % DYN_MEL_MULTIPLE)
    T_mel_batch = min(T_mel_batch, WHISPER_MAX_MEL_FRAMES)

    feats = torch.zeros(B, N_MELS, T_mel_batch, dtype=torch.float32)
    for i, f in enumerate(feats_list):
        L = min(f.shape[1], T_mel_batch)
        feats[i, :, :L] = f[:, :L]

    il = torch.tensor(real_lens, dtype=torch.long)
    tl = torch.tensor([l.shape[0] for l in labs], dtype=torch.long)
    return feats, torch.cat(labs), il, tl


trl = DataLoader(WhisperAudioDataset('train', feature_extractor),
                  batch_size=BS, shuffle=True, collate_fn=collate,
                  num_workers=4, pin_memory=True, drop_last=True,
                  persistent_workers=True, prefetch_factor=4)
# PLAN A: Val subset 1/3 -- 22K val 7.6K'ya iner, epoch basi ~30 dk kazanc.
_val_full = WhisperAudioDataset('val', feature_extractor)
_val_subset = torch.utils.data.Subset(_val_full, list(range(0, len(_val_full), 3)))
print(f'  Val subset: {len(_val_full)} -> {len(_val_subset)} (1/3)')
vll = DataLoader(_val_subset,
                  batch_size=BS, shuffle=False, collate_fn=collate,
                  num_workers=4, pin_memory=True,
                  persistent_workers=True, prefetch_factor=4)
print(f'Train: {len(trl)} batch | Val: {len(vll)} batch')
print(f'Cache: {USE_MEL_CACHE}, SpecAugment: {USE_SPECAUGMENT}, SpeedPerturb: {USE_SPEED_PERTURB}')


## 4) Egitim setup — Optimizer, scheduler, metrics, resume

In [ ]:
MDIR = '/kaggle/working/ckpt_audio_whisper_ctc'
UDIR = '/kaggle/working/upload_audio_whisper_ctc'
os.makedirs(MDIR, exist_ok=True)
os.makedirs(UDIR, exist_ok=True)
CP = f'{MDIR}/checkpoint.pt'
BP = f'{MDIR}/best_model.pt'

# Kaggle credentials
KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', 'UNKNOWN')
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    KAGGLE_USER = _us.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY'] = _us.get_secret('KAGGLE_KEY')
except Exception as e:
    print(f'UYARI: Secrets okunamadi: {e}')

DATASET_SLUG = 'lippy-audio-whisper-ctc-v2-checkpoint'
DATASET_ID = f'{KAGGLE_USER}/{DATASET_SLUG}'
with open(f'{UDIR}/dataset-metadata.json', 'w') as f:
    json.dump({
        'title': 'LipPY Audio Whisper CTC v2 (medium + SpecAugment + Speed Perturb)',
        'id': DATASET_ID,
        'licenses': [{'name': 'CC0-1.0'}],
    }, f)
print(f'Upload: {DATASET_ID}')


def greedy_ctc(lp):
    ids = lp.argmax(-1).tolist()
    d = []
    p = -1
    for i in ids:
        if i != p and i != BLANK_IDX:
            d.append(i)
        p = i
    return d


def edit_dist(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for ai in a:
        r = [dp[0] + 1]
        for j, bj in enumerate(b):
            r.append(min(r[-1] + 1, dp[j + 1] + 1, dp[j] + (0 if ai == bj else 1)))
        dp = r
    return dp[-1]


def per_calc(preds, refs):
    td = tl = 0
    for p, r in zip(preds, refs):
        td += edit_dist(p, r)
        tl += max(len(r), 1)
    return td / max(tl, 1)


def metrics(preds, refs):
    per_all = per_calc(preds, refs)
    preds_ph = [[t for t in p if t != SPACE_IDX] for p in preds]
    refs_ph = [[t for t in r if t != SPACE_IDX] for r in refs]
    per_phoneme = per_calc(preds_ph, refs_ph)
    tp = fp = fn = 0
    for p, r in zip(preds, refs):
        ps = sum(1 for t in p if t == SPACE_IDX)
        rs = sum(1 for t in r if t == SPACE_IDX)
        tp += min(ps, rs); fp += max(0, ps - rs); fn += max(0, rs - ps)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-6)
    return {'per_all': per_all, 'per_phoneme': per_phoneme, 'space_f1': f1}


def cosine_warmup(ep, warm, total, min_r=0.05):
    """FIX (v2): Eski versiyonda ep=0 icin max(0.01, 0/warm)=0.01 -> LR_FE 3e-8
    olup ep1 fiilen bos geciyordu. (ep+1)/warm ve floor 0.1 ile ep1=0.5x LR,
    ep2=1.0x LR (peak). min_r 0.01 -> 0.05 son epoch'larda LR olu kalmasin."""
    if ep < warm:
        return max(0.1, (ep + 1) / warm)
    p = (ep - warm) / max(1, total - warm)
    return max(min_r, 0.5 * (1 + math.cos(math.pi * p)))


def upload(ep):
    for f in glob.glob(f'{MDIR}/*.pt'):
        shutil.copy(f, f'{UDIR}/{os.path.basename(f)}')
    r = subprocess.run(['kaggle', 'datasets', 'version', '-p', UDIR,
                         '-m', f'whisper-ctc ep{ep}', '--dir-mode', 'zip'],
                       capture_output=True, text=True, timeout=1800)
    if r.returncode == 0:
        print(f'[upload ep{ep}] VERSION OK')
    else:
        stderr = (r.stderr or '').lower()
        if 'not found' in stderr or '404' in stderr:
            cr = subprocess.run(['kaggle', 'datasets', 'create', '-p', UDIR,
                                  '--dir-mode', 'zip'],
                                 capture_output=True, text=True, timeout=1800)
            print(f'[upload ep{ep}] CREATE rc={cr.returncode}')
        else:
            print(f'[upload ep{ep}] FAIL rc={r.returncode}')


In [ ]:
model = WhisperAudioModel().to(DEVICE)

# Optimizer 2 grup
fe_p = [p for n, p in model.named_parameters()
        if p.requires_grad and n.startswith('encoder.')]
be_p = [p for n, p in model.named_parameters()
        if p.requires_grad and not n.startswith('encoder.')]

opt = AdamW([
    {'params': fe_p, 'lr': LR_FE},
    {'params': be_p, 'lr': LR_BE},
], betas=(0.9, 0.98), weight_decay=0.01)

sched = LambdaLR(opt, [
    lambda e: cosine_warmup(e, WARMUP, EPOCHS),
    lambda e: cosine_warmup(e, WARMUP, EPOCHS),
])

ctc_crit = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
scaler = torch.amp.GradScaler('cuda')

print(f'FE params (unfrozen encoder): {sum(p.numel() for p in fe_p):,}')
print(f'BE params (CTC head):         {sum(p.numel() for p in be_p):,}')

# Resume
start_ep = 1
best_per = 1.0
best_f1 = 0.0

if not FORCE_SCRATCH:
    ckpts = ([CP] if os.path.exists(CP) else []) + glob.glob(
        f'/kaggle/input/**/{DATASET_SLUG}/**/checkpoint.pt', recursive=True)
    ckpts = sorted(set(ckpts), key=lambda p: -os.path.getmtime(p) if os.path.exists(p) else 0)
    if ckpts:
        print(f'Resume candidate: {ckpts[0]}')
        try:
            ck = torch.load(ckpts[0], map_location=DEVICE, weights_only=False)
            cand_per = ck.get('best_per', 1.0)
            cand_ep = ck.get('epoch', 0)
            if cand_per > MAX_RESUME_PER:
                print(f'UYARI: best_per={cand_per:.1%} > MAX, ATLANIYOR')
            else:
                model.load_state_dict(ck['model_state_dict'])
                start_ep = cand_ep + 1
                best_per = cand_per
                best_f1 = ck.get('best_f1', 0.0)
                try:
                    opt.load_state_dict(ck['opt'])
                    sched.load_state_dict(ck['sched'])
                except Exception:
                    pass
                if 'scaler' in ck:
                    scaler.load_state_dict(ck['scaler'])
                print(f'Resume OK: epoch {cand_ep} (best_per={best_per:.1%})')
        except Exception as e:
            print(f'Resume HATA: {e}')


## 5) Egitim Loop

AMP fp16, gradient accumulation, her epoch best ckpt save, her 2 ep upload.

**Beklenen hiz (PLAN A.2: 10 epoch + Whisper-medium + freeze 18 + BS=4 + mel cache):**
~1.2-1.5h/epoch -> **12-15h toplam, 2 Kaggle session, ~1 gun calendar.**

- Mel cache: dataloader bottleneck cozuldu (PLAN A medium %1.26h/epoch idi)
- Medium compute (24 layer x 1024 dim x 80 mel bin) -- large-v3'ten 2x hizli
- BS=4 + ACCUM=4: effective batch 16, T4 16GB rahat
- Session limit 12h: ep1 > 2h ise notebook 2 epoch yapip auto-pause + resume
- 10 epoch / 2 session = ortalama 5 epoch/session

**VRAM (medium + BS=4):** ~3.1GB model + activations, T4 16GB'da rahat.

**PER hedef (PLAN A.2):** %27-30 (PLAN A %32.1'den -2 ila -5 puan, ilimli SpecAug ile).
**Karar:**
  - PER ≤ %30 → v3 fusion icin bu ckpt'yi kullan
  - PER > %30 → eski PLAN A %32.1 ckpt'sine geri don, v3 ile o ile dene

In [ ]:
print(f'\n--- Audio Whisper-CTC v2 Egitim (PLAN B.3: large-v3, start_ep={start_ep}) ---\n')
epoch_history = []

# FIX (v2): Dinamik session limit
# Kaggle session ~12h, buffer 1h. Eger ep1 > 2h ise bu session sadece 2 epoch
# yap, resume ile devam (uzun training'i bolup checkpoint guvenligi)
SESSION_EPOCH_LIMIT = EPOCHS  # baslangic: limit yok
SLOW_EPOCH_THRESHOLD_SEC = 2 * 3600

for epoch in range(start_ep, EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    tl_s = tc = nan = oom = 0
    opt.zero_grad(set_to_none=True)

    for step, (feats, tgts, il, tl) in enumerate(trl):
        feats = feats.to(DEVICE)
        tgts = tgts.to(DEVICE)
        il_dev = il.to(DEVICE)

        # PLAN B.3 OOM FIX: backward'ı da try-except'e al (worst case batch OOM crash etmesin)
        try:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                lp = model(feats)  # (T, B, V)
            T_out = lp.shape[0]
            il_clamp = il_dev.clamp(max=T_out)
            loss = ctc_crit(lp.float(), tgts, il_clamp, tl) / ACCUM

            if torch.isnan(loss) or torch.isinf(loss):
                nan += 1
                # Skip backward, ama opt step koordinasyonu icin zero_grad
                if (step + 1) % ACCUM == 0:
                    opt.zero_grad(set_to_none=True)
                continue

            scaler.scale(loss).backward()
        except RuntimeError as e:
            if 'out of memory' in str(e):
                oom += 1
                torch.cuda.empty_cache()
                # Backward yarim kalmis olabilir, opt state korumak icin zero_grad
                opt.zero_grad(set_to_none=True)
                continue
            raise

        if (step + 1) % ACCUM == 0 or (step + 1) == len(trl):
            try:
                scaler.unscale_(opt)
                gn = nn.utils.clip_grad_norm_(model.parameters(), GCLIP)
                if torch.isfinite(gn):
                    scaler.step(opt)
                    tl_s += loss.item() * ACCUM
                    tc += 1
                else:
                    nan += 1
                scaler.update()
            except RuntimeError as e:
                if 'out of memory' in str(e):
                    oom += 1
                    torch.cuda.empty_cache()
                else:
                    raise
            opt.zero_grad(set_to_none=True)

    # Val
    model.eval()
    preds = []
    refs = []
    vl_s = vc = 0
    with torch.no_grad():
        for feats, tgts, il, tl in vll:
            feats = feats.to(DEVICE)
            tgts = tgts.to(DEVICE)  # FIX (v2): Train ile tutarli, CTC loss icin gerekli
            il_dev = il.to(DEVICE)
            try:
                # FIX (v2): Val'da autocast -> fp16 forward, %40-50 hizlanma
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    lp = model(feats)
            except RuntimeError as e:
                if 'out of memory' in str(e):
                    torch.cuda.empty_cache()
                    continue
                raise
            T_out = lp.shape[0]
            il_clamp = il_dev.clamp(max=T_out)
            try:
                vl = ctc_crit(lp.float(), tgts, il_clamp, tl)
                if not (torch.isnan(vl) or torch.isinf(vl)):
                    vl_s += vl.item(); vc += 1
            except Exception:
                pass
            for i in range(feats.shape[0]):
                preds.append(greedy_ctc(lp[:il_clamp[i], i]))
            off = 0
            tgts_cpu = tgts.cpu()
            for i in range(feats.shape[0]):
                refs.append(tgts_cpu[off:off + tl[i]].tolist())
                off += tl[i].item()

    train_loss = tl_s / max(tc, 1)
    val_loss = vl_s / max(vc, 1)
    m = metrics(preds, refs)

    if m['per_phoneme'] < best_per:
        # FIX (v2): per_phoneme (SPACE haric) -> AV-Fusion downstream icin
        # daha dogru checkpoint kriteri. per_all SPACE'i de sayar, large-v3
        # SPACE'i kolay ogrenir -> per_all bias'li olabilir.
        best_per = m['per_phoneme']
        torch.save(model.state_dict(), BP)
        print(f'  [BEST] PER-phoneme {best_per:.1%} kaydedildi')
    if m['space_f1'] > best_f1:
        best_f1 = m['space_f1']

    sched.step()
    ns = f' NaN:{nan}' if nan else ''
    os_ = f' OOM:{oom}' if oom else ''
    gap = train_loss - val_loss
    ep_sec = int(time.time() - t0)
    print(f'Ep {epoch:2d}/{EPOCHS} | TrL:{train_loss:.3f} | VaL:{val_loss:.3f} | '
          f'gap:{gap:+.2f} | PER-all:{m["per_all"]:.1%} ph:{m["per_phoneme"]:.1%} | '
          f'SP F1:{m["space_f1"]:.2f} | {ep_sec}s{ns}{os_}')

    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        for i in range(min(2, len(preds))):
            print(f'  H: "{phonemes_to_text(refs[i])[:80]}"')
            print(f'  T: "{phonemes_to_text(preds[i])[:80]}"')

    epoch_history.append({
        'epoch': epoch, 'per_all': m['per_all'], 'per_phoneme': m['per_phoneme'],
        'space_f1': m['space_f1'], 'val_loss': val_loss,
    })

    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'opt': opt.state_dict(), 'sched': sched.state_dict(),
        'scaler': scaler.state_dict(),
        'best_per': best_per, 'best_f1': best_f1,
    }, CP)

    # FIX (v2): Ep1 sonrasi runtime budget kontrolu
    # Ep1 > 2h ise bu session sadece 2 epoch (current + 1) yap, sonra durup upload+exit
    if epoch == start_ep and ep_sec > SLOW_EPOCH_THRESHOLD_SEC:
        SESSION_EPOCH_LIMIT = start_ep + 1
        print(f'  [BUDGET] ep{epoch} {ep_sec}s (>2h) -> bu session ep{SESSION_EPOCH_LIMIT}\'de durur, resume ile devam')

    if epoch % UPLOAD_EVERY == 0:
        upload(epoch)

    # FIX (v2): Session limit -> upload + break
    if epoch >= SESSION_EPOCH_LIMIT:
        upload(epoch)
        print(f'\n[BUDGET] Session limit ep{epoch} -> cikiliyor. Notebook yeniden basla, resume otomatik.')
        break

# Final upload (yalniz tum epochlar tamamlandiysa)
if epoch >= EPOCHS:
    upload(EPOCHS)

print('\n' + '=' * 70)
print('AUDIO WHISPER-CTC v2 EGITIM SONUC (PLAN B.3: large-v3)')
print('=' * 70)
print(f'{"Epoch":>6} | {"PER-all":>8} | {"PER-ph":>8} | {"SP F1":>6} | {"VaL":>6}')
print('-' * 50)
for h in epoch_history:
    print(f'{h["epoch"]:>6} | {h["per_all"]:>7.1%} | {h["per_phoneme"]:>7.1%} | '
          f'{h["space_f1"]:>6.2f} | {h["val_loss"]:>6.3f}')
print('-' * 50)
print(f'{"BEST":>6} | {best_per:>7.1%} (per_phoneme)')

print(f'\nKarsilastir:')
print(f'  Whisper-medium CTC v1 (baseline):   PER 31.2%')
print(f'  Whisper-large-v3 CTC v2 (PLAN B.3): PER {best_per:.1%}  (per_phoneme)')
print(f'  Iyilestirme:                        {(0.312 - best_per)*100:+.1f} puan vs v1 medium')
print(f'\nKarar:')
print(f'  - PER <= %26: PLAN B.3 basarili, AV-Fusion v2\'de large-v3 kullan')
print(f'  - PER > %26:  Daha agresif fix gerek (freeze 16, 15 epoch, vs.)')
